# RL-LLM: Reinforcement Learning for Language Model Training

This notebook implements a hierarchical RL-based language model using PPO (Proximal Policy Optimization).

**Week 5-6 Implementation:**
- Multi-component reward system (fluency, coherence, task completion, safety)
- Task-specific environments (Q&A, Conversation)
- Hierarchical policy (High-level: intentions, Low-level: tokens)
- Hierarchical PPO training

**Goal:** Generate coherent text for simple tasks like question-answering and conversation

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install torch>=2.0.0
!pip install transformers>=4.30.0
!pip install datasets>=2.14.0
!pip install numpy>=1.24.0
!pip install tqdm>=4.65.0

In [ ]:
# Mixed Precision Training Setup (CRITICAL for memory reduction)
from torch.cuda.amp import autocast, GradScaler

# Initialize gradient scaler for mixed precision
scaler = GradScaler()

print("✓ Mixed precision training enabled (will save ~50% GPU memory)")

✓ Mixed precision training enabled (will save ~50% GPU memory)


/tmp/ipython-input-936088801.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

PyTorch version: 2.8.0+cu126
CUDA available: True
CUDA version: 12.6
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 85.17 GB


## 2. Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from datasets import load_dataset
from tqdm import tqdm
import numpy as np
import random
import math
import re
import os
from typing import Dict, List, Tuple, Optional, Any
from io import StringIO
import sys
import signal
from contextlib import contextmanager

## 3. Utility Functions

In [ ]:
# Training utilities
def set_seed(seed: int = 42):
    """Set random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_device():
    """Get appropriate device (CUDA if available)"""
    return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Set seed for reproducibility
set_seed(42)
device = get_device()
print(f"Using device: {device}")

Using device: cuda


## 4. Dataset Loader Classes

In [ ]:
class TimeoutException(Exception):
    """Custom exception for code execution timeout"""
    pass

@contextmanager
def time_limit(seconds):
    """Context manager for enforcing execution timeout"""
    def signal_handler(signum, frame):
        raise TimeoutException("Timed out!")

    signal.signal(signal.SIGALRM, signal_handler)
    signal.alarm(seconds)
    try:
        yield
    finally:
        signal.alarm(0)

In [ ]:
# CELL 4 (REPLACEMENT): Programming Dataset Loaders

class HumanEvalDataset:
    """HumanEval dataset for code generation"""

    def __init__(self, split: str = 'test', use_subset: Optional[int] = None):
        print("Loading HumanEval dataset...")
        try:
            from datasets import load_dataset
            dataset = load_dataset("openai_humaneval", split=split)
        except Exception as e:
            print(f"Error loading HumanEval: {e}")
            dataset = []

        self.problems = []
        for example in dataset:
            problem = {
                'task_id': example['task_id'],
                'prompt': example['prompt'],
                'test': example['test'],
                'entry_point': example['entry_point']
            }
            self.problems.append(problem)

        if use_subset is not None:
            self.problems = self.problems[:use_subset]

        print(f"Loaded {len(self.problems)} code problems")

    def get_random_problem(self) -> Dict:
        return random.choice(self.problems)

    def get_problem_by_id(self, task_id: str) -> Optional[Dict]:
        for p in self.problems:
            if p['task_id'] == task_id:
                return p
        return None

    def evaluate_code(self, code: str, problem: Dict) -> Tuple[bool, int, Optional[str]]:
        """Execute code and check if tests pass"""
        try:
            # Combine generated code with test cases
            full_code = code + '\n\n' + problem['test'] + '\n\n'
            full_code += f"check({problem['entry_point']})\n"

            # Execute with timeout (5 seconds)
            old_stdout = sys.stdout
            old_stderr = sys.stderr
            redirected_output = StringIO()
            sys.stdout = redirected_output
            sys.stderr = redirected_output

            try:
                exec(full_code, {})
                sys.stdout = old_stdout
                sys.stderr = old_stderr
                return (True, 1, None)
            except Exception as e:
                sys.stdout = old_stdout
                sys.stderr = old_stderr
                return (False, 0, str(e))
        except Exception as e:
            return (False, 0, str(e))

    def compute_reward(self, code: str, problem: Dict) -> float:
        """Compute reward from code execution"""
        passed, total, error = self.evaluate_code(code, problem)

        if passed:
            return 10.0
        elif error and "SyntaxError" in str(error):
            return -5.0
        elif error and "Timeout" in str(error):
            return -3.0
        else:
            return -1.0

    def __len__(self):
        return len(self.problems)

    def __getitem__(self, idx):
        return self.problems[idx]


class TheStackDataset:
    """The Stack dataset - filtered for Python code"""

    def __init__(self, language: str = 'python', num_samples: int = 1000):
        print(f"Loading The Stack dataset ({language})...")
        try:
            from datasets import load_dataset
            # Load a streaming subset
            dataset = load_dataset(
                "bigcode/the-stack-dedup",
                data_dir=f"data/{language}",
                split="train",
                streaming=True
            )

            # Take first num_samples
            self.problems = []
            for i, example in enumerate(dataset):
                if i >= num_samples:
                    break

                # Extract code snippet
                code = example['content']

                # Create a "complete the function" task
                # Split code into prompt and completion
                lines = code.split('\n')
                if len(lines) > 10:
                    split_point = len(lines) // 2
                    prompt = '\n'.join(lines[:split_point])
                    expected = '\n'.join(lines[split_point:])

                    self.problems.append({
                        'task_id': f'stack_{i}',
                        'prompt': prompt,
                        'expected': expected,
                        'full_code': code
                    })

            print(f"Loaded {len(self.problems)} code samples")
        except Exception as e:
            print(f"Error loading The Stack: {e}")
            print("You may need to authenticate with HuggingFace Hub")
            self.problems = []

    def get_random_problem(self) -> Dict:
        if not self.problems:
            return {'task_id': 'empty', 'prompt': 'def hello():', 'expected': '\n    return "world"'}
        return random.choice(self.problems)

    def compute_reward(self, generated: str, problem: Dict) -> float:
        """Reward based on similarity to expected completion"""
        expected = problem.get('expected', '')

        # Simple token overlap metric
        gen_tokens = set(generated.split())
        exp_tokens = set(expected.split())

        if not exp_tokens:
            return -1.0

        overlap = len(gen_tokens & exp_tokens) / len(exp_tokens)

        # Reward based on overlap
        if overlap > 0.8:
            return 10.0
        elif overlap > 0.5:
            return 5.0
        elif overlap > 0.3:
            return 2.0
        else:
            return -1.0

    def __len__(self):
        return len(self.problems)

    def __getitem__(self, idx):
        return self.problems[idx]


class CodeChainDataset:
    """CodeChain dataset for chain-of-thought code reasoning"""

    def __init__(self, num_samples: int = 500):
        print("Loading CodeChain dataset...")
        try:
            from datasets import load_dataset
            dataset = load_dataset("Elfsong/CodeChain", split="train")

            self.problems = []
            for i, example in enumerate(dataset):
                if i >= num_samples:
                    break

                problem = {
                    'task_id': f'codechain_{i}',
                    'prompt': example.get('question', ''),
                    'solution': example.get('solution', ''),
                    'chain': example.get('chain_of_thought', '')
                }
                self.problems.append(problem)

            print(f"Loaded {len(self.problems)} CodeChain problems")
        except Exception as e:
            print(f"Error loading CodeChain: {e}")
            self.problems = []

    def get_random_problem(self) -> Dict:
        if not self.problems:
            return {'task_id': 'empty', 'prompt': 'Write a function to add two numbers'}
        return random.choice(self.problems)

    def compute_reward(self, generated: str, problem: Dict) -> float:
        """Reward based on solution similarity"""
        solution = problem.get('solution', '')

        # Check if key elements of solution appear in generated code
        if not solution:
            return 0.0

        # Simple substring matching
        if solution.lower() in generated.lower():
            return 10.0

        # Partial credit for similar tokens
        gen_tokens = set(generated.lower().split())
        sol_tokens = set(solution.lower().split())

        if sol_tokens:
            overlap = len(gen_tokens & sol_tokens) / len(sol_tokens)
            return overlap * 10.0 - 2.0

        return -1.0

    def __len__(self):
        return len(self.problems)

    def __getitem__(self, idx):
        return self.problems[idx]


class RedPajamaCodeDataset:
    """RedPajama dataset - code subset"""

    def __init__(self, num_samples: int = 1000):
        print("Loading RedPajama Code dataset...")
        try:
            from datasets import load_dataset
            # Load GitHub subset of RedPajama
            dataset = load_dataset(
                "togethercomputer/RedPajama-Data-1T-Sample",
                split="train",
                streaming=True
            )

            self.problems = []
            count = 0

            for example in dataset:
                if count >= num_samples:
                    break

                # Filter for code content (GitHub source)
                if example.get('meta', {}).get('redpajama_set_name') == 'RedPajamaGithub':
                    text = example['text']

                    # Create completion task
                    if len(text) > 100:
                        split_point = len(text) // 2
                        prompt = text[:split_point]
                        expected = text[split_point:]

                        self.problems.append({
                            'task_id': f'redpajama_{count}',
                            'prompt': prompt,
                            'expected': expected
                        })
                        count += 1

            print(f"Loaded {len(self.problems)} RedPajama code samples")
        except Exception as e:
            print(f"Error loading RedPajama: {e}")
            self.problems = []

    def get_random_problem(self) -> Dict:
        if not self.problems:
            return {'task_id': 'empty', 'prompt': 'import numpy as np\n\ndef '}
        return random.choice(self.problems)

    def compute_reward(self, generated: str, problem: Dict) -> float:
        """Reward based on completion quality"""
        expected = problem.get('expected', '')

        # Token overlap
        gen_tokens = set(generated.split())
        exp_tokens = set(expected.split())

        if not exp_tokens:
            return -1.0

        overlap = len(gen_tokens & exp_tokens) / len(exp_tokens)

        if overlap > 0.7:
            return 10.0
        elif overlap > 0.4:
            return 5.0
        else:
            return -1.0

    def __len__(self):
        return len(self.problems)

    def __getitem__(self, idx):
        return self.problems[idx]


# Dataset factory function
def create_code_dataset(dataset_type: str = 'humaneval', **kwargs):
    """
    Factory function to create code datasets

    Args:
        dataset_type: 'humaneval', 'stack', 'codechain', or 'redpajama'
        **kwargs: Additional arguments for dataset constructor
    """
    if dataset_type == 'humaneval':
        return HumanEvalDataset(**kwargs)
    elif dataset_type == 'stack':
        return TheStackDataset(**kwargs)
    elif dataset_type == 'codechain':
        return CodeChainDataset(**kwargs)
    elif dataset_type == 'redpajama':
        return RedPajamaCodeDataset(**kwargs)
    else:
        raise ValueError(f"Unknown dataset type: {dataset_type}")


# Test datasets
print("\n" + "="*70)
print("Testing Dataset Loaders")
print("="*70)

# Test HumanEval
try:
    humaneval = create_code_dataset('humaneval', use_subset=5)
    sample = humaneval.get_random_problem()
    print(f"\nHumanEval sample:")
    print(f"  Task: {sample['task_id']}")
    print(f"  Prompt: {sample['prompt'][:100]}...")
except Exception as e:
    print(f"HumanEval loading failed: {e}")

print("\n✓ Dataset loaders ready!")


Testing Dataset Loaders
Loading HumanEval dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

openai_humaneval/test-00000-of-00001.par(…):   0%|          | 0.00/83.9k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Loaded 5 code problems

HumanEval sample:
  Task: HumanEval/0
  Prompt: from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
  ...

✓ Dataset loaders ready!


In [ ]:
# NEW CELL 5: Code Generation Environment

class CodeGenerationEnvironment:
    """Specialized RL environment for code generation tasks"""

    def __init__(self, tokenizer, dataset, max_length: int = 512):
        self.tokenizer = tokenizer
        self.dataset = dataset
        self.max_length = max_length

        self.current_problem = None
        self.current_sequence = []
        self.step_count = 0
        self.done = False

        self.eos_token_id = tokenizer.eos_token_id

    def reset(self) -> Dict:
        """Reset with a random code problem"""
        self.current_problem = self.dataset.get_random_problem()

        # Tokenize the prompt
        prompt = self.current_problem['prompt']
        token_ids = self.tokenizer.encode(prompt, return_tensors='pt')[0]

        self.current_sequence = token_ids.tolist()
        self.step_count = 0
        self.done = False

        return self.get_state()

    def step(self, action: int) -> Tuple[Dict, float, bool, Dict]:
        """Take a step by generating a token"""
        self.current_sequence.append(action)
        self.step_count += 1

        text = self.tokenizer.decode(self.current_sequence)

        # Termination conditions for code
        self.done = (
            self.step_count >= self.max_length or
            action == self.eos_token_id or
            '\n\n\n' in text[-20:]  # Multiple blank lines = code complete
        )

        if self.done:
            # Extract generated code (remove prompt)
            prompt_length = len(self.tokenizer.encode(self.current_problem['prompt']))
            generated_tokens = self.current_sequence[prompt_length:]
            generated_code = self.tokenizer.decode(generated_tokens)

            # Compute reward using dataset's evaluation
            reward = self.dataset.compute_reward(generated_code, self.current_problem)
        else:
            # Small step penalty
            reward = -0.01

        next_state = self.get_state()

        info = {
            'text': text,
            'length': len(self.current_sequence),
            'problem_id': self.current_problem['task_id']
        }

        return next_state, reward, self.done, info

    def get_state(self) -> Dict:
        return {
            'token_ids': torch.tensor(self.current_sequence),
            'step': self.step_count,
            'done': self.done
        }

## 5. Neural Network Architectures

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for transformers"""

    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(1)]

In [ ]:
class HierarchicalPolicy(nn.Module):
    """Two-level hierarchical policy for structured generation"""

    def __init__(self, vocab_size: int, d_model: int = 256, intention_dim: int = 64,
                 num_layers: int = 4, nhead: int = 4, max_len: int = 512):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.intention_dim = intention_dim
        self.max_len = max_len # Store max_len

        # Shared state encoder
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_len) # Pass max_len

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            batch_first=True
        )
        self.state_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers // 2)

        # HIGH-LEVEL POLICY: State -> Intention distribution
        self.intention_mean = nn.Linear(d_model, intention_dim)
        self.intention_logstd = nn.Linear(d_model, intention_dim)

        # LOW-LEVEL POLICY: State + Intention -> Token distribution
        self.token_decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=d_model * 4,
                batch_first=True
            ),
            num_layers=num_layers // 2
        )

        self.intention_proj = nn.Linear(intention_dim, d_model)
        self.output_layer = nn.Linear(d_model, vocab_size)

        # Separate value heads
        self.high_value = nn.Linear(d_model, 1)
        self.low_value = nn.Linear(d_model, 1)

    def encode_state(self, token_ids):
        # Truncate input sequence to max_len for positional encoding
        seq_len = token_ids.size(1)
        if seq_len > self.max_len:
            token_ids = token_ids[:, :self.max_len]
            seq_len = self.max_len

        x = self.embedding(token_ids) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        x = self.state_encoder(x)
        return x

    def sample_intention(self, state_encoding):
        pooled = state_encoding.mean(dim=1)
        mean = self.intention_mean(pooled)
        logstd = self.intention_logstd(pooled)
        std = torch.exp(logstd)
        dist = torch.distributions.Normal(mean, std)
        intention = dist.rsample()
        return intention, dist

    def forward(self, token_ids, intention=None, return_full_output=False):
        state_encoding = self.encode_state(token_ids)

        if intention is None:
            intention, intention_dist = self.sample_intention(state_encoding)
        else:
            intention_dist = None

        intention_encoded = self.intention_proj(intention)
        intention_expanded = intention_encoded.unsqueeze(1).expand(
            -1, state_encoding.size(1), -1
        )

        # Ensure target sequence length for decoder matches encoder output length
        decoded = self.token_decoder(state_encoding, intention_expanded)
        logits = self.output_layer(decoded[:, -1, :])

        logits = torch.clamp(logits, min=-100, max=100)


        if return_intention:
            pooled = state_encoding.mean(dim=1)
            high_value = self.high_value(pooled)
            low_value = self.low_value(pooled)

            return {
                'logits': logits,
                'intention': intention,
                'intention_dist': intention_dist,
                'high_value': high_value,
                'low_value': low_value
            }
        else:
            return logits

    def get_action_distribution(self, token_ids, intention=None):
        logits = self.forward(token_ids, intention)
        return torch.distributions.Categorical(logits=logits)



## 5.5 Week 7: Energy-Based Value Functions

**Key Innovation:** Unified Energy-Based Model that:
- Assigns energy scores to text (lower energy = better quality)
- Serves as value function for RL
- Generates text by finding low-energy configurations
- Learns text evaluation and generation simultaneously

In [ ]:
# NEW CODE CELL - Energy Network Architecture
class EnergyBasedValueNetwork(nn.Module):
    """
    Energy-based model that assigns energy scores to text sequences.
    Lower energy = higher quality text.
    This network serves dual purposes:
    1. Value estimation for RL (energy → value)
    2. Text quality evaluation for generation
    """

    def __init__(self, vocab_size: int, d_model: int = 256, num_layers: int = 4,
                 nhead: int = 4, max_len: int = 512):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_len = max_len

        # Shared text encoder
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Energy computation heads (multi-scale)
        self.token_energy = nn.Linear(d_model, 1)  # Token-level energy
        self.sequence_energy = nn.Linear(d_model, 1)  # Sequence-level energy

        # Contrastive learning projection for structural quality
        self.projection_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, d_model // 2)
        )

    def encode_sequence(self, token_ids):
        """Encode token sequence into latent representations"""
        seq_len = token_ids.size(1)
        if seq_len > self.max_len:
            token_ids = token_ids[:, :self.max_len]

        x = self.embedding(token_ids) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        encoded = self.encoder(x)
        return encoded

    def compute_energy(self, token_ids):
        """
        Compute energy score for a sequence.
        Returns lower energy for higher quality text.
        """
        encoded = self.encode_sequence(token_ids)

        # Token-level energies (local coherence)
        token_energies = self.token_energy(encoded).squeeze(-1)

        # Sequence-level energy (global coherence)
        pooled = encoded.mean(dim=1)
        seq_energy = self.sequence_energy(pooled)

        # Combined energy (normalized)
        total_energy = token_energies.mean(dim=1, keepdim=True) + seq_energy

        return {
            'total_energy': total_energy,
            'token_energies': token_energies,
            'sequence_energy': seq_energy,
            'encoded': encoded,
            'pooled': pooled
        }

    def energy_to_value(self, energy):
        """Convert energy to value estimate for RL (negative energy = positive value)"""
        return -energy['total_energy']

    def compute_contrastive_loss(self, encoded_batch, temperature=0.1):
        """
        Contrastive learning to discover text structure.
        Similar texts should have similar encodings.
        """
        # Project to contrastive space
        projections = self.projection_head(encoded_batch)
        projections = F.normalize(projections, dim=-1)

        # Compute similarity matrix
        similarity = torch.matmul(projections, projections.t()) / temperature

        # Contrastive loss (InfoNCE)
        batch_size = encoded_batch.size(0)
        labels = torch.arange(batch_size, device=encoded_batch.device)

        loss = F.cross_entropy(similarity, labels)
        return loss


class EnergyBasedGenerator:
    """
    Text generation by finding low-energy configurations.
    Uses energy network to guide generation towards high-quality text.
    """

    def __init__(self, energy_network, tokenizer, device):
        self.energy_net = energy_network
        self.tokenizer = tokenizer
        self.device = device

    def generate_low_energy(self, prompt_ids, max_length=50, num_candidates=5,
                           temperature=1.0, energy_weight=1.0):
        """
        Generate text by sampling and selecting low-energy continuations.

        Args:
            prompt_ids: Initial token sequence
            max_length: Maximum generation length
            num_candidates: Number of candidate tokens to consider per step
            temperature: Sampling temperature
            energy_weight: Weight for energy-based selection
        """
        self.energy_net.eval()

        current_seq = prompt_ids.clone()

        with torch.no_grad():
            for _ in range(max_length):
                # Generate candidate tokens
                candidates = []
                energies = []

                for _ in range(num_candidates):
                    # Sample next token (with temperature)
                    candidate = torch.randint(
                        0, self.tokenizer.vocab_size, (1,), device=self.device
                    )

                    # Compute energy for candidate continuation
                    test_seq = torch.cat([current_seq, candidate.unsqueeze(0)], dim=1)
                    energy = self.energy_net.compute_energy(test_seq)

                    candidates.append(candidate)
                    energies.append(energy['total_energy'].item())

                # Select candidate with lowest energy
                best_idx = np.argmin(energies)
                next_token = candidates[best_idx]

                current_seq = torch.cat([current_seq, next_token.unsqueeze(0)], dim=1)

                # Stop if EOS token
                if next_token.item() == self.tokenizer.eos_token_id:
                    break

        return current_seq

print("✓ Energy-Based Value Network implemented")

✓ Energy-Based Value Network implemented


In [ ]:
# NEW CODE CELL - Energy-Based Training Integration
class EnergyAugmentedHierarchicalPolicy(HierarchicalPolicy):
    """
    Hierarchical Policy augmented with Energy-Based Value Functions.
    Combines hierarchical RL with energy-based text evaluation.
    """

    def __init__(self, vocab_size: int, d_model: int = 256, intention_dim: int = 64,
                 num_layers: int = 4, nhead: int = 4, max_len: int = 512):
        super().__init__(vocab_size, d_model, intention_dim, num_layers, nhead, max_len)

        # Add energy-based value network
        self.energy_network = EnergyBasedValueNetwork(
            vocab_size=vocab_size,
            d_model=d_model,
            num_layers=num_layers,
            nhead=nhead,
            max_len=max_len
        )

        # Fusion layer to combine hierarchical and energy-based values
        self.value_fusion = nn.Sequential(
            nn.Linear(3, 64),  # 3 values: high, low, energy
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, token_ids, intention=None, return_full_output=False,
                compute_energy=True):
        # Standard hierarchical forward pass
        state_encoding = self.encode_state(token_ids)

        if intention is None:
            intention, intention_dist = self.sample_intention(state_encoding)
        else:
            intention_dist = None

        intention_encoded = self.intention_proj(intention)
        intention_expanded = intention_encoded.unsqueeze(1).expand(
            -1, state_encoding.size(1), -1
        )

        decoded = self.token_decoder(state_encoding, intention_expanded)
        logits = self.output_layer(decoded[:, -1, :])
        logits = torch.clamp(logits, min=-100, max=100)

        if return_intention:
            pooled = state_encoding.mean(dim=1)
            high_value = self.high_value(pooled)
            low_value = self.low_value(pooled)

            # Compute energy-based value
            if compute_energy:
                energy_result = self.energy_network.compute_energy(token_ids)
                energy_value = self.energy_network.energy_to_value(energy_result)

                # Fuse values
                stacked_values = torch.cat([high_value, low_value, energy_value], dim=1)
                fused_value = self.value_fusion(stacked_values)
            else:
                energy_result = None
                fused_value = (high_value + low_value) / 2

            return {
                'logits': logits,
                'intention': intention,
                'intention_dist': intention_dist,
                'high_value': high_value,
                'low_value': low_value,
                'energy_value': energy_value if compute_energy else None,
                'fused_value': fused_value,
                'energy_result': energy_result
            }
        else:
            return logits

print("✓ Energy-Augmented Hierarchical Policy implemented")

✓ Energy-Augmented Hierarchical Policy implemented


## 5.6 Week 8: Multi-Scale Temporal Learning

**Multi-Scale Architecture:**
1. **Short-term (1-5 tokens):** Word-to-word dependencies
2. **Medium-term (5-20 tokens):** Sentence-level coherence  
3. **Long-term (20-100 tokens):** Document structure
4. **Structural:** Grammar and syntax patterns

**Key Innovation:** Hierarchical attention across different time scales


In [ ]:
# NEW CODE CELL - Multi-Scale Temporal Module
class MultiScaleTemporalModule(nn.Module):
    """
    Process text at multiple time scales simultaneously.
    Captures short-term, medium-term, and long-term dependencies.
    """

    def __init__(self, d_model: int = 256, nhead: int = 4):
        super().__init__()

        self.d_model = d_model

        # Short-term: Local attention (window size 5)
        self.short_term_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            batch_first=True
        )

        # Medium-term: Sentence-level attention (window size 20)
        self.medium_term_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            batch_first=True
        )

        # Long-term: Document-level attention (full sequence)
        self.long_term_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            batch_first=True
        )

        # Structural: Pattern detection
        self.pattern_detector = nn.Sequential(
            nn.Conv1d(d_model, d_model, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(d_model, d_model, kernel_size=5, padding=2),
            nn.ReLU()
        )

        # Scale fusion
        self.scale_fusion = nn.Sequential(
            nn.Linear(d_model * 4, d_model * 2),
            nn.ReLU(),
            nn.Linear(d_model * 2, d_model)
        )

        # Layer norms
        self.norm_short = nn.LayerNorm(d_model)
        self.norm_medium = nn.LayerNorm(d_model)
        self.norm_long = nn.LayerNorm(d_model)
        self.norm_struct = nn.LayerNorm(d_model)

    def create_windowed_mask(self, seq_len, window_size, device):
        """Create attention mask for windowed attention"""
        mask = torch.ones(seq_len, seq_len, device=device) * float('-inf')
        for i in range(seq_len):
            start = max(0, i - window_size // 2)
            end = min(seq_len, i + window_size // 2 + 1)
            mask[i, start:end] = 0
        return mask

    def forward(self, x):
        """
        Args:
            x: (batch_size, seq_len, d_model)
        Returns:
            Multi-scale representation
        """
        batch_size, seq_len, d_model = x.shape
        device = x.device

        # Short-term processing (local context)
        short_mask = self.create_windowed_mask(seq_len, window_size=5, device=device)
        short_out, _ = self.short_term_attn(x, x, x, attn_mask=short_mask)
        short_out = self.norm_short(x + short_out)

        # Medium-term processing (sentence-level)
        medium_mask = self.create_windowed_mask(seq_len, window_size=20, device=device)
        medium_out, _ = self.medium_term_attn(x, x, x, attn_mask=medium_mask)
        medium_out = self.norm_medium(x + medium_out)

        # Long-term processing (full sequence)
        long_out, _ = self.long_term_attn(x, x, x)
        long_out = self.norm_long(x + long_out)

        # Structural processing (pattern detection)
        x_transpose = x.transpose(1, 2)  # (batch, d_model, seq_len)
        struct_out = self.pattern_detector(x_transpose)
        struct_out = struct_out.transpose(1, 2)  # Back to (batch, seq_len, d_model)
        struct_out = self.norm_struct(x + struct_out)

        # Fuse all scales
        multi_scale = torch.cat([short_out, medium_out, long_out, struct_out], dim=-1)
        fused = self.scale_fusion(multi_scale)

        return {
            'fused': fused,
            'short_term': short_out,
            'medium_term': medium_out,
            'long_term': long_out,
            'structural': struct_out
        }


class MultiScaleContrastiveLearning(nn.Module):
    """
    Learn text structure through contrastive learning at multiple scales.
    Discovers grammar and linguistic patterns without explicit supervision.
    """

    def __init__(self, d_model: int = 256):
        super().__init__()

        # Projection heads for different scales
        self.short_proj = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, 128)
        )

        self.medium_proj = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, 128)
        )

        self.long_proj = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Linear(d_model, 128)
        )

    def compute_contrastive_loss(self, embeddings, scale='short', temperature=0.1):
        """
        Contrastive loss for learning representations at different scales
        """
        # Select appropriate projection
        if scale == 'short':
            proj = self.short_proj
        elif scale == 'medium':
            proj = self.medium_proj
        else:
            proj = self.long_proj

        # Project and normalize
        z = proj(embeddings)
        z = F.normalize(z, dim=-1)

        # Compute similarity matrix
        batch_size = z.size(0)
        similarity = torch.matmul(z, z.t()) / temperature

        # InfoNCE loss
        labels = torch.arange(batch_size, device=z.device)
        loss = F.cross_entropy(similarity, labels)

        return loss

print("✓ Multi-Scale Temporal Module implemented")

✓ Multi-Scale Temporal Module implemented


In [ ]:
# NEW CODE CELL - Complete Integrated Architecture
class FullRLLMModel(nn.Module):
    """
    Complete RL-LLM model integrating:
    - Week 5-6: Hierarchical Policy
    - Week 7: Energy-Based Value Functions
    - Week 8: Multi-Scale Temporal Learning
    """

    def __init__(self, vocab_size: int, d_model: int = 256, intention_dim: int = 64,
                 num_layers: int = 4, nhead: int = 4, max_len: int = 512):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_len = max_len

        # Base components
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_len)

        # Week 8: Multi-scale temporal processing
        self.multi_scale_module = MultiScaleTemporalModule(d_model, nhead)
        self.contrastive_learner = MultiScaleContrastiveLearning(d_model)

        # Week 5-6: Hierarchical policy components
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            batch_first=True
        )
        self.state_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers // 2)

        # High-level policy (intentions)
        self.intention_mean = nn.Linear(d_model, intention_dim)
        self.intention_logstd = nn.Linear(d_model, intention_dim)

        # Low-level policy (tokens)
        self.token_decoder = nn.TransformerDecoder(
            nn.TransformerDecoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=d_model * 4,
                batch_first=True
            ),
            num_layers=num_layers // 2
        )
        self.intention_proj = nn.Linear(intention_dim, d_model)
        self.output_layer = nn.Linear(d_model, vocab_size)

        # Week 7: Energy-based value network
        self.energy_network = EnergyBasedValueNetwork(
            vocab_size, d_model, num_layers, nhead, max_len
        )

        # Value heads
        self.high_value = nn.Linear(d_model, 1)
        self.low_value = nn.Linear(d_model, 1)
        self.multi_scale_value = nn.Linear(d_model, 1)

        # Final value fusion
        self.value_fusion = nn.Sequential(
            nn.Linear(4, 64),  # 4 values: high, low, energy, multi-scale
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def encode_with_multiscale(self, token_ids):
        """Encode sequence with multi-scale processing"""
        seq_len = token_ids.size(1)
        if seq_len > self.max_len:
            token_ids = token_ids[:, :self.max_len]

        # Initial embedding
        x = self.embedding(token_ids) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)

        # Multi-scale temporal processing
        multi_scale_output = self.multi_scale_module(x)

        # Further encoding
        encoded = self.state_encoder(multi_scale_output['fused'])

        return {
            'encoded': encoded,
            'multi_scale': multi_scale_output
        }

    def forward(self, token_ids, intention=None, return_full_output=False):
        """
        Forward pass with full integration of all components
        """
        # Multi-scale encoding
        encoding_output = self.encode_with_multiscale(token_ids)
        state_encoding = encoding_output['encoded']
        multi_scale = encoding_output['multi_scale']

        # Sample intention (high-level policy)
        if intention is None:
            pooled = state_encoding.mean(dim=1)
            mean = self.intention_mean(pooled)
            logstd = self.intention_logstd(pooled)
            std = torch.exp(logstd)
            dist = torch.distributions.Normal(mean, std)
            intention = dist.rsample()
            intention_dist = dist
        else:
            intention_dist = None

        # Decode with intention (low-level policy)
        intention_encoded = self.intention_proj(intention)
        intention_expanded = intention_encoded.unsqueeze(1).expand(
            -1, state_encoding.size(1), -1
        )
        decoded = self.token_decoder(state_encoding, intention_expanded)
        logits = self.output_layer(decoded[:, -1, :])
        logits = torch.clamp(logits, min=-100, max=100)

        if return_full_output:
            pooled = state_encoding.mean(dim=1)
            multi_scale_pooled = multi_scale['fused'].mean(dim=1)

            # All value estimates
            high_value = self.high_value(pooled)
            low_value = self.low_value(pooled)
            multi_scale_value = self.multi_scale_value(multi_scale_pooled)

            # Energy-based value
            energy_result = self.energy_network.compute_energy(token_ids)
            energy_value = self.energy_network.energy_to_value(energy_result)

            # Fuse all values
            stacked_values = torch.cat([
                high_value, low_value, energy_value, multi_scale_value
            ], dim=1)
            fused_value = self.value_fusion(stacked_values)

            return {
                'logits': logits,
                'intention': intention,
                'intention_dist': intention_dist,
                'high_value': high_value,
                'low_value': low_value,
                'energy_value': energy_value,
                'multi_scale_value': multi_scale_value,
                'fused_value': fused_value,
                'multi_scale_output': multi_scale,
                'energy_result': energy_result
            }
        else:
            return logits

print("✓ Complete integrated RL-LLM model implemented")

✓ Complete integrated RL-LLM model implemented


## 6. Reward Functions

In [ ]:
class MultiComponentReward:
    """Multi-component reward system with fluency, coherence, task completion, and safety"""

    def __init__(self, tokenizer=None, task_evaluator=None):
        self.tokenizer = tokenizer
        self.task_evaluator = task_evaluator

        # Load GPT-2 for fluency scoring
        print("Loading GPT-2 for fluency evaluation...")
        self.fluency_device = device
        self.fluency_tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
        self.fluency_model = GPT2LMHeadModel.from_pretrained('gpt2-medium').to(self.fluency_device)
        self.fluency_model.eval()

        if hasattr(self.fluency_model, 'gradient_checkpointing_enable'):
            self.fluency_model.gradient_checkpointing_enable()

        if self.fluency_tokenizer.pad_token is None:
            self.fluency_tokenizer.pad_token = self.fluency_tokenizer.eos_token

        self.unsafe_patterns = [
            'kill', 'hate', 'violence', 'attack', 'bomb', 'weapon',
            'racist', 'sexist', 'discriminat'
        ]

    def compute_fluency(self, token_ids: torch.Tensor) -> float:
        if len(token_ids) < 2:
            return 0.0

        with torch.no_grad():
            if token_ids.dim() == 1:
                token_ids = token_ids.unsqueeze(0)
            token_ids = token_ids.to(self.fluency_device)
            outputs = self.fluency_model(token_ids, labels=token_ids)
            loss = outputs.loss.item()
            return -loss * 0.1

    def compute_coherence(self, token_ids: torch.Tensor, text: str = None) -> float:
        if len(token_ids) < 2:
            return 0.0

        token_list = token_ids.tolist() if torch.is_tensor(token_ids) else token_ids
        unique_tokens = len(set(token_list))
        total_tokens = len(token_list)
        diversity_ratio = unique_tokens / total_tokens if total_tokens > 0 else 0.0
        coherence_score = diversity_ratio * 0.2

        immediate_repeats = sum(1 for i in range(len(token_list) - 1)
                               if token_list[i] == token_list[i+1])
        repetition_penalty = -0.05 * immediate_repeats

        return coherence_score + repetition_penalty

    def compute_task_completion(self, token_ids: torch.Tensor, text: str = None,
                                prompt: str = None, target: str = None) -> float:
        if self.task_evaluator is not None:
            return self.task_evaluator(text, prompt, target)

        score = 0.0
        length = len(token_ids)
        if 5 <= length <= 100:
            score += 0.1
        elif length < 5:
            score -= 0.2
        elif length > 200:
            score -= 0.1

        return score

    def compute_safety(self, token_ids: torch.Tensor, text: str = None) -> float:
        if text is None and self.tokenizer is not None:
            text = self.tokenizer.decode(token_ids)

        if text is None:
            return 0.0

        text_lower = text.lower()
        for pattern in self.unsafe_patterns:
            if pattern in text_lower:
                return -1.0

        return 0.0

    def compute_reward(self, token_ids: torch.Tensor, text: str = None,
                      prompt: str = None, target: str = None) -> float:
        if text is None and self.tokenizer is not None:
            text = self.tokenizer.decode(token_ids)

        fluency = self.compute_fluency(token_ids)
        coherence = self.compute_coherence(token_ids, text)
        task_completion = self.compute_task_completion(token_ids, text, prompt, target)
        safety = self.compute_safety(token_ids, text)

        total_reward = (
            0.4 * fluency +
            0.2 * coherence +
            0.3 * task_completion +
            0.1 * safety
        )

        if safety < 0:
            total_reward += safety * 5.0

        return total_reward

## 7. Task-Specific Environments

## 8. Rollout Buffer

In [ ]:
class RolloutBuffer:
    """Experience buffer for storing and processing trajectories"""

    def __init__(self):
        self.clear()

    def clear(self):
        self.states = []
        self.actions = []
        self.rewards = []
        self.log_probs = []
        self.values = []
        self.dones = []
        self.intentions = []

    def add(self, state, action, reward, log_prob, value, done, intention=None):
        self.states.append(state['token_ids'])
        self.actions.append(action)
        self.rewards.append(reward)
        self.log_probs.append(log_prob)
        self.values.append(value)
        self.dones.append(done)

        if intention is not None:
            self.intentions.append(intention)

    def get_batches(self, gamma=0.99, gae_lambda=0.95):
        rewards = np.array(self.rewards)
        values = np.array([v.item() if torch.is_tensor(v) else v for v in self.values])
        dones = np.array(self.dones, dtype=np.float32)

        # Compute advantages using GAE
        advantages = np.zeros_like(rewards)
        last_advantage = 0

        for t in reversed(range(len(rewards))):
            if t == len(rewards) - 1:
                next_value = 0
            else:
                next_value = values[t + 1]

            delta = rewards[t] + gamma * next_value * (1 - dones[t]) - values[t]
            advantages[t] = last_advantage = delta + gamma * gae_lambda * (1 - dones[t]) * last_advantage

        returns = advantages + values
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        # Pad states
        max_len = max(len(s) for s in self.states)
        padded_states = []
        for s in self.states:
            if len(s) < max_len:
                pad_len = max_len - len(s)
                s_padded = torch.cat([s, torch.zeros(pad_len, dtype=s.dtype)])
            else:
                s_padded = s
            padded_states.append(s_padded)

        batch = {
            'states': torch.stack(padded_states),
            'actions': torch.tensor(self.actions, dtype=torch.long),
            'log_probs': torch.tensor(self.log_probs, dtype=torch.float32),
            'returns': torch.tensor(returns, dtype=torch.float32),
            'advantages': torch.tensor(advantages, dtype=torch.float32),
        }

        if len(self.intentions) > 0:
            batch['intentions'] = torch.stack(self.intentions)

        return batch

    def __len__(self):
        return len(self.rewards)

## 9. Hierarchical PPO Trainer

In [ ]:
def sanitize_logits(logits, eps=1e-8):
    """Remove NaN/Inf from logits and clamp to reasonable range"""
    logits = torch.where(torch.isnan(logits), torch.zeros_like(logits), logits)
    logits = torch.where(torch.isinf(logits),
                        torch.sign(logits) * 1e4 * torch.ones_like(logits),
                        logits)
    logits = torch.clamp(logits, min=-20.0, max=20.0)
    return logits

class HierarchicalPPOTrainer:
    """PPO trainer for hierarchical two-level policy"""

    def __init__(self, hierarchical_policy, lr: float = 3e-4, clip_ratio: float = 0.2,
                 value_coef: float = 0.5, entropy_coef: float = 0.1, intention_coef: float = 0.1):
        self.policy = hierarchical_policy
        self.clip_ratio = clip_ratio
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef
        self.intention_coef = intention_coef

        self.optimizer = optim.Adam(hierarchical_policy.parameters(), lr=lr)
        self.device = next(hierarchical_policy.parameters()).device

    def compute_hierarchical_policy_loss(self, states, actions, old_log_probs,
                                         old_intentions, advantages):
        outputs = self.policy(states, return_full_output=True)

        # Low-level policy loss
        #action_dist = torch.distributions.Categorical(logits=outputs['logits'])

        logits = sanitize_logits(outputs['logits'])
        action_dist = torch.distributions.Categorical(logits=logits)
        new_log_probs = action_dist.log_prob(actions)

        ratio = torch.exp(new_log_probs - old_log_probs)
        clipped_ratio = torch.clamp(ratio, 1 - self.clip_ratio, 1 + self.clip_ratio)

        low_level_loss = -torch.min(
            ratio * advantages,
            clipped_ratio * advantages
        ).mean()

        token_entropy = action_dist.entropy().mean()

        # High-level policy loss
        if outputs['intention_dist'] is not None:
            old_intention_log_prob = outputs['intention_dist'].log_prob(old_intentions).sum(dim=-1)
            intention_entropy = outputs['intention_dist'].entropy().sum(dim=-1).mean()
            high_level_loss = -old_intention_log_prob.mean()
        else:
            high_level_loss = torch.tensor(0.0, device=self.device)
            intention_entropy = torch.tensor(0.0, device=self.device)

        policy_loss = low_level_loss + self.intention_coef * high_level_loss
        total_entropy = token_entropy + 0.1 * intention_entropy

        return policy_loss, total_entropy, high_level_loss

    def compute_hierarchical_value_loss(self, states, returns):
        outputs = self.policy(states, return_full_output=True)

        high_value = outputs['high_value'].squeeze(-1)
        low_value = outputs['low_value'].squeeze(-1)

        high_value_loss = nn.functional.mse_loss(high_value, returns)
        low_value_loss = nn.functional.mse_loss(low_value, returns)

        return (high_value_loss + low_value_loss) / 2.0

    def update(self, buffer, epochs: int = 4):
        batches = buffer.get_batches()

        states = batches['states'].to(self.device)
        actions = batches['actions'].to(self.device)
        old_log_probs = batches['log_probs'].to(self.device)
        returns = batches['returns'].to(self.device)
        advantages = batches['advantages'].to(self.device)

        if 'intentions' in batches:
            old_intentions = batches['intentions'].to(self.device)
        else:
            with torch.no_grad():
                outputs = self.policy(states, return_full_output=True)
                old_intentions = outputs['intention']

        stats = {
            'policy_loss': 0.0,
            'value_loss': 0.0,
            'entropy': 0.0,
            'intention_loss': 0.0
        }

        for epoch in range(epochs):
            with autocast():

                policy_loss, entropy, intention_loss = self.compute_hierarchical_policy_loss(
                    states, actions, old_log_probs, old_intentions, advantages
                )
                value_loss = self.compute_hierarchical_value_loss(states, returns)

                total_loss = policy_loss + self.value_coef * value_loss - self.entropy_coef * entropy

                self.optimizer.zero_grad()

            # Use mixed precision for backward pass
            scaler.scale(total_loss).backward()
            scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 0.2)
            scaler.step(self.optimizer)
            scaler.update()

            stats['policy_loss'] += policy_loss.item()
            stats['value_loss'] += value_loss.item()
            stats['entropy'] += entropy.item()
            stats['intention_loss'] += intention_loss.item()

        for key in stats:
            stats[key] /= epochs

        return stats

    def collect_episode_with_intentions(self, env, buffer, max_steps: int = 100):
        state = env.reset()
        done = False
        episode_reward = 0.0
        steps = 0

        while not done and steps < max_steps:
            token_ids = state['token_ids'].unsqueeze(0).to(self.device)

            with torch.no_grad():
                outputs = self.policy(token_ids, return_full_output=True)
                action_dist = torch.distributions.Categorical(logits=outputs['logits'])
                action = action_dist.sample()
                log_prob = action_dist.log_prob(action)
                intention = outputs['intention']
                value = outputs['low_value']

            next_state, reward, done, info = env.step(action.item())

            buffer.add(
                state=state,
                action=action.item(),
                reward=reward,
                log_prob=log_prob.item(),
                value=value.item(),
                done=done,
                intention=intention.cpu()
            )

            state = next_state
            episode_reward += reward
            steps += 1

        return episode_reward

In [ ]:
# NEW CODE CELL - Enhanced PPO Trainer with Energy and Multi-Scale
class EnhancedPPOTrainer(HierarchicalPPOTrainer):
    """
    PPO Trainer enhanced with:
    - Energy-based value training
    - Multi-scale contrastive learning
    - Structural regularization
    """

    def __init__(self, policy, lr=5e-4, energy_weight=0.3, contrastive_weight=0.2):
        super().__init__(policy, lr)
        self.energy_weight = energy_weight
        self.contrastive_weight = contrastive_weight

    def train_step(self, buffer, epochs=4, batch_size=64):
        """Enhanced training step with energy and multi-scale objectives"""
        batch = buffer.get_batches()

        stats = {
            'policy_loss': 0.0,
            'value_loss': 0.0,
            'entropy': 0.0,
            'intention_loss': 0.0,
            'energy_loss': 0.0,
            'contrastive_loss': 0.0
        }

        for _ in range(epochs):
            # Prepare batch
            states = batch['states'].to(self.device)
            actions = batch['actions'].to(self.device)
            old_log_probs = batch['log_probs'].to(self.device)
            returns = batch['returns'].to(self.device)
            advantages = batch['advantages'].to(self.device)

            # Forward pass with full output
            with autocast():
                outputs = self.policy(states, return_full_output=True)

                # Policy loss (PPO)
                logits = sanitize_logits(outputs['logits'])
                action_dist = torch.distributions.Categorical(logits=logits)
                new_log_probs = action_dist.log_prob(actions)
                ratio = torch.exp(new_log_probs - old_log_probs)

                surr1 = ratio * advantages
                surr2 = torch.clamp(ratio, 0.8, 1.2) * advantages
                policy_loss = -torch.min(surr1, surr2).mean()

                # Value loss (MSE for fused value)
                value_loss = F.mse_loss(outputs['fused_value'].squeeze(), returns)

                # Entropy bonus
                entropy = action_dist.entropy().mean()

                # Intention regularization
                if outputs['intention_dist'] is not None:
                    prior = torch.distributions.Normal(0, 1)
                    intention_loss = torch.distributions.kl_divergence(
                        outputs['intention_dist'], prior
                    ).mean()
                else:
                    intention_loss = torch.tensor(0.0, device=self.device)

                # Energy-based loss (train energy network)
                energy_loss = -outputs['energy_value'].mean()  # Maximize value = minimize energy

                # Multi-scale contrastive loss
                multi_scale_out = outputs['multi_scale_output']
                contrastive_loss = 0.0

                for scale_name in ['short_term', 'medium_term', 'long_term']:
                    scale_embeddings = multi_scale_out[scale_name].mean(dim=1)
                    contrastive_loss += self.policy.contrastive_learner.compute_contrastive_loss(
                        scale_embeddings, scale=scale_name.split('_')[0]
                    )
                contrastive_loss /= 3

                # Total loss
                total_loss = (
                    policy_loss +
                    0.5 * value_loss -
                    0.01 * entropy +
                    0.01 * intention_loss +
                    self.energy_weight * energy_loss +
                    self.contrastive_weight * contrastive_loss
                )

            # Backward pass
            self.optimizer.zero_grad()
            scaler.scale(total_loss).backward()
            scaler.unscale_(self.optimizer)
            torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 0.5)
            scaler.step(self.optimizer)
            scaler.update()

            # Update stats
            stats['policy_loss'] += policy_loss.item()
            stats['value_loss'] += value_loss.item()
            stats['entropy'] += entropy.item()
            stats['intention_loss'] += intention_loss.item()
            stats['energy_loss'] += energy_loss.item()
            stats['contrastive_loss'] += contrastive_loss.item()

        # Average over epochs
        for key in stats:
            stats[key] /= epochs

        return stats

print("✓ Enhanced PPO Trainer with Week 7-8 components implemented")

✓ Enhanced PPO Trainer with Week 7-8 components implemented


## 10. Initialize Components

In [ ]:
# # Load tokenizer
# print("Loading tokenizer...")
# tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
# tokenizer.pad_token = tokenizer.eos_token

# # Create dataset
# print("Creating dataset...")
# dataset = TinyDataset()
# print(f"Dataset size: {len(dataset)}")

# # Create environment
# print("Creating Q&A environment...")
# env = QuestionAnsweringEnvironment(
#     tokenizer=tokenizer,
#     dataset=dataset,
#     max_length=100
# )

# # Create reward function
# print("Creating multi-component reward function...")
# reward_fn = MultiComponentReward(tokenizer=tokenizer)

# print("\n✓ All components initialized!")

In [ ]:
# CELL 10 (REPLACEMENT): Initialize Components for Code Generation

# Configuration - Choose your dataset
DATASET_TYPE = 'humaneval'  # Options: 'humaneval', 'stack', 'codechain', 'redpajama'
MAX_LENGTH = 512  # Longer for code generation - Aligned with PositionalEncoding max_len

print(f"Initializing components for {DATASET_TYPE.upper()} dataset...")
print("="*70)

# Load tokenizer
print("\n1. Loading tokenizer...")
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

# Create code dataset
print(f"\n2. Creating {DATASET_TYPE} dataset...")
if DATASET_TYPE == 'humaneval':
    dataset = create_code_dataset('humaneval', use_subset=20)  # Start with subset
elif DATASET_TYPE == 'stack':
    dataset = create_code_dataset('stack', language='python', num_samples=100)
elif DATASET_TYPE == 'codechain':
    dataset = create_code_dataset('codechain', num_samples=100)
elif DATASET_TYPE == 'redpajama':
    dataset = create_code_dataset('redpajama', num_samples=100)

print(f"   Dataset size: {len(dataset)}")

# Create code generation environment
print("\n3. Creating Code Generation environment...")
env = CodeGenerationEnvironment(
    tokenizer=tokenizer,
    dataset=dataset,
    max_length=MAX_LENGTH # Use the consistent MAX_LENGTH
)

# Create multi-component reward function
print("\n4. Creating multi-component reward function...")
reward_fn = MultiComponentReward(tokenizer=tokenizer)

print("\n✓ All components initialized for code generation!")
print("="*70)

Initializing components for HUMANEVAL dataset...

1. Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...


2. Creating humaneval dataset...
Loading HumanEval dataset...
Loaded 20 code problems
   Dataset size: 20

3. Creating Code Generation environment...

4. Creating multi-component reward function...
Loading GPT-2 for fluency evaluation...


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


✓ All components initialized for code generation!


## 11. Create Hierarchical Policy

In [ ]:
# Create full RL-LLM model with Week 7-8 components
print("Creating full RL-LLM model with Week 7-8 components...")
vocab_size = tokenizer.vocab_size

# ✅ Use the FullRLLMModel that integrates everything
policy = FullRLLMModel(
    vocab_size=vocab_size,
    d_model=128,
    intention_dim=32,
    num_layers=2,
    nhead=4,
    max_len=1024  # Aligned with config
).to(device)

# Enable gradient checkpointing for memory efficiency
if hasattr(policy, 'gradient_checkpointing_enable'):
    policy.gradient_checkpointing_enable()

# Manually enable for sub-modules
if hasattr(policy, 'state_encoder'):
    for module in policy.state_encoder.modules():
        if hasattr(module, 'gradient_checkpointing'):
            module.gradient_checkpointing = True

if hasattr(policy, 'token_decoder'):
    for module in policy.token_decoder.modules():
        if hasattr(module, 'gradient_checkpointing'):
            module.gradient_checkpointing = True

print(f"Policy created with {sum(p.numel() for p in policy.parameters()):,} parameters")

# ✅ Use the EnhancedPPOTrainer with energy and multi-scale
print("Creating enhanced PPO trainer with energy and multi-scale learning...")
trainer = EnhancedPPOTrainer(
    policy,
    lr=3e-4,  # Slightly higher learning rate
    energy_weight=0.3,
    contrastive_weight=0.2
)

print("\n✓ Full RL-LLM model ready!")
# print("  ✓ Hierarchical Policy (Week 5-6)")
# print("  ✓ Energy-Based Value Functions (Week 7)")
# print("  ✓ Multi-Scale Temporal Learning (Week 8)")

Creating full RL-LLM model with Week 7-8 components...
Policy created with 20,840,407 parameters
Creating enhanced PPO trainer with energy and multi-scale learning...

✓ Full RL-LLM model ready!


## 12. Training Loop

In [ ]:
# CELL 12 (REPLACEMENT): Training Configuration for Code Generation

# Training configuration
NUM_ITERATIONS = 500  # More iterations for code tasks
EPISODES_PER_ITERATION = 32  # Fewer episodes (code generation is slower)
MAX_LENGTH = 1024  # Longer sequences for code - Aligned with PositionalEncoding max_len
LOG_INTERVAL = 100  # Log less frequently

print(f"\n{'='*70}")
print("Starting Hierarchical Policy Training on Code Generation")
print(f"{'='*70}")
print(f"Dataset: {DATASET_TYPE.upper()}")
print(f"Iterations: {NUM_ITERATIONS}")
print(f"Episodes per iteration: {EPISODES_PER_ITERATION}")
print(f"Max length: {MAX_LENGTH}")
print(f"{'='*70}\n")

best_reward = float('-inf')
training_history = []

for iteration in tqdm(range(NUM_ITERATIONS), desc="Training"):
    policy.train()
    buffer = RolloutBuffer()
    episode_rewards = []

    # Collect episodes
    for episode in range(EPISODES_PER_ITERATION):
        episode_reward = trainer.collect_episode_with_intentions(
            env=env,
            buffer=buffer,
            max_steps=MAX_LENGTH # Use the consistent MAX_LENGTH
        )
        episode_rewards.append(episode_reward)

    # Update policy
    if len(buffer) > 0:
        stats = trainer.update(buffer, epochs=50)
    else:
        stats = {}

    # Logging
    avg_reward = sum(episode_rewards) / len(episode_rewards)
    training_history.append(avg_reward)

    if (iteration + 1) % LOG_INTERVAL == 0:
        print(f"\n{'='*70}")
        print(f"Iteration {iteration + 1}/{NUM_ITERATIONS}")
        print(f"{'='*70}")
        print(f"  Avg Training Reward: {avg_reward:.2f}")
        print(f"  Policy Loss: {stats.get('policy_loss', 0):.4f}")
        print(f"  Value Loss: {stats.get('value_loss', 0):.4f}")
        print(f"  Entropy: {stats.get('entropy', 0):.4f}")
        print(f"  Intention Loss: {stats.get('intention_loss', 0):.4f}")

        # Generate sample code
        policy.eval()
        state = env.reset()
        done = False
        steps = 0

        while not done and steps < 100:
            token_ids = state['token_ids'].unsqueeze(0).to(device)
            with torch.no_grad():
                outputs = policy(token_ids, return_full_output=True)
                action_dist = torch.distributions.Categorical(logits=outputs['logits'])
                action = action_dist.sample()
            state, _, done, info = env.step(action.item())
            steps += 1

        print(f"\n  Sample Generated Code:")
        print(f"  {'-'*66}")
        # Show first 200 chars of generated code
        code_snippet = info['text'][:200].replace('\n', '\n  ')
        print(f"  {code_snippet}...")
        print(f"  {'-'*66}")

    # Save best model
    if avg_reward > best_reward:
        best_reward = avg_reward
        print(f"\n  ✓ New best model! Reward: {best_reward:.2f}")

print(f"\n{'='*70}")
print("Training Complete!")
print(f"{'='*70}")
print(f"Best reward achieved: {best_reward:.2f}")


Starting Hierarchical Policy Training on Code Generation
Dataset: HUMANEVAL
Iterations: 500
Episodes per iteration: 8
Max length: 256



Training:   0%|          | 0/500 [00:00<?, ?it/s]/tmp/ipython-input-4250933238.py:94: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Training:   0%|          | 1/500 [00:59<8:16:07, 59.65s/it]


  ✓ New best model! Reward: -2.56


Training:   2%|▏         | 9/500 [08:23<7:32:32, 55.30s/it]


Iteration 10/500
  Avg Training Reward: -2.56
  Policy Loss: 5.9733
  Value Loss: 0.0030
  Entropy: 16.9930
  Intention Loss: 60.4014


...


Iteration 60/500
  Avg Training Reward: -2.56
  Policy Loss: 14.3528
  Value Loss: 0.0033
  Entropy: 25.3839
  Intention Loss: 144.2528


Training:  12%|█▏        | 60/500 [55:30<6:49:20, 55.82s/it]


  Sample Generated Code:
  ------------------------------------------------------------------
  from typing import List
  
  
  def has_close_elements(numbers: List[float], threshold: float) -> bool:
      """ Check if in given list of numbers, are any two numbers closer to each other than
      given thr...
  ------------------------------------------------------------------


Training:  14%|█▍        | 69/500 [1:04:35<6:43:27, 56.17s/it]


KeyboardInterrupt: 

## 13. Visualize Training Progress

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
plt.plot(training_history)
plt.xlabel('Iteration')
plt.ylabel('Average Reward')
plt.title('Training Progress: Hierarchical Policy on Q&A Task')
plt.grid(True, alpha=0.3)
plt.show()

# Compute moving average
window = 10
if len(training_history) >= window:
    moving_avg = np.convolve(training_history, np.ones(window)/window, mode='valid')
    plt.figure(figsize=(12, 6))
    plt.plot(training_history, alpha=0.3, label='Raw')
    plt.plot(range(window-1, len(training_history)), moving_avg, label=f'{window}-iteration Moving Average')
    plt.xlabel('Iteration')
    plt.ylabel('Average Reward')
    plt.title('Training Progress with Moving Average')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

## 14. Final Evaluation

In [ ]:
def evaluate_policy(policy, env, tokenizer, num_episodes=10):
    """Evaluate trained policy"""
    policy.eval()

    total_reward = 0.0
    successful_episodes = 0
    generations = []

    for _ in range(num_episodes):
        state = env.reset()
        done = False
        episode_reward = 0.0

        while not done:
            token_ids = state['token_ids'].unsqueeze(0).to(device)

            with torch.no_grad():
                outputs = policy(token_ids, return_full_output=True)
                action_dist = torch.distributions.Categorical(logits=outputs['logits'])
                action = action_dist.sample()

            state, reward, done, info = env.step(action.item())
            episode_reward += reward

        total_reward += episode_reward
        if episode_reward > 0:
            successful_episodes += 1

        generations.append(info['text'])

    return {
        'avg_reward': total_reward / num_episodes,
        'success_rate': successful_episodes / num_episodes,
        'generations': generations
    }

print(f"\n{'='*70}")
print("Final Evaluation")
print(f"{'='*70}\n")

eval_results = evaluate_policy(policy, env, tokenizer, num_episodes=10)

print(f"Average Reward: {eval_results['avg_reward']:.2f}")
print(f"Success Rate: {eval_results['success_rate']:.1%}")
print(f"\nSample Generations:\n")

for i, gen in enumerate(eval_results['generations'][:5]):
    print(f"{i+1}. {gen}")
    print()

## 15. Test Specific Questions

In [ ]:
# def test_question(policy, tokenizer, question, max_steps=50):
#     """Test policy on a specific question"""
#     policy.eval()

#     # Create temporary environment for this question
#     test_env = QuestionAnsweringEnvironment(tokenizer, dataset=None, max_length=100)
#     state = test_env.reset(question=question)

#     done = False
#     steps = 0

#     while not done and steps < max_steps:
#         token_ids = state['token_ids'].unsqueeze(0).to(device)

#         with torch.no_grad():
#             outputs = policy(token_ids, return_full_output=True)
#             action_dist = torch.distributions.Categorical(logits=outputs['logits'])
#             action = action_dist.sample()

#         state, reward, done, info = test_env.step(action.item())
#         steps += 1

#     return info['text']

# # Test on custom questions
# test_questions = [
#     "What is 7+3?",
#     "What is 12-5?",
#     "What is the capital of France?",
#     "Who invented the telephone?",
#     "What color is the sky?"
# ]

# print(f"\n{'='*70}")
# print("Testing on Custom Questions")
# print(f"{'='*70}\n")

# for question in test_questions:
#     answer = test_question(policy, tokenizer, question)
#     print(f"Q: {question}")
#     print(f"A: {answer}")
#     print()

In [ ]:
# NEW CELL: Test Code Generation

def test_code_generation(policy, tokenizer, dataset, num_samples=5):
    """Test policy on code generation tasks"""
    policy.eval()

    print(f"\n{'='*70}")
    print(f"Testing Code Generation on {num_samples} problems")
    print(f"{'='*70}\n")

    for i in range(num_samples):
        problem = dataset.get_random_problem()

        # Create temp environment
        temp_env = CodeGenerationEnvironment(tokenizer, dataset, max_length=512)
        temp_env.current_problem = problem

        # Generate
        prompt = problem['prompt']
        token_ids = tokenizer.encode(prompt, return_tensors='pt')[0]
        state = {'token_ids': token_ids, 'step': 0, 'done': False}

        done = False
        steps = 0

        while not done and steps < 200:
            token_ids = state['token_ids'].unsqueeze(0).to(device)
            with torch.no_grad():
                outputs = policy(token_ids, return_full_output=True)
                action_dist = torch.distributions.Categorical(logits=outputs['logits'])
                action = action_dist.sample()

            # Manual step
            current_seq = state['token_ids'].tolist()
            current_seq.append(action.item())
            text = tokenizer.decode(current_seq)

            done = steps >= 200 or action.item() == tokenizer.eos_token_id
            state = {'token_ids': torch.tensor(current_seq), 'step': steps, 'done': done}
            steps += 1

        # Extract generated code
        prompt_len = len(tokenizer.encode(prompt))
        generated_code = tokenizer.decode(state['token_ids'][prompt_len:])

        # Show results
        print(f"Problem {i+1}: {problem['task_id']}")
        print(f"Prompt:\n{prompt[:150]}...")
        print(f"\nGenerated Code:\n{generated_code[:300]}")
        print(f"\n{'-'*70}\n")

# Run test
test_code_generation(policy, tokenizer, dataset, num_samples=3)

## 16. Save Model

In [ ]:
save_path = 'full_rllm_model_week_7_8.pt'  # ✅ New descriptive name

torch.save({
    'policy_state_dict': policy.state_dict(),
    'best_reward': best_reward,
    'training_history': training_history,
    'vocab_size': vocab_size,
    'd_model': 256,
    'intention_dim': 64,
    'num_layers': 4,
    'nhead': 4,
    'max_len': 512,  # ✅ Added
    'model_type': 'FullRLLMModel',  # ✅ Added - Important for loading
    'week': '7-8'  # ✅ Added - Track implementation version
}, save_path)

print(f"✓ Full model (Week 7-8) saved to {save_path}")

# Download to local machine
from google.colab import files
files.download(save_path)
print(f"✓ Model downloaded!")

## 17. Load Saved Model (Optional)

In [ ]:
# To load a saved model
def load_model(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Recreate model with saved architecture
    loaded_policy = HierarchicalPolicy(
        vocab_size=checkpoint['vocab_size'],
        d_model=checkpoint['d_model'],
        intention_dim=checkpoint['intention_dim'],
        num_layers=checkpoint['num_layers'],
        nhead=checkpoint['nhead']
    ).to(device)

    # Load weights
    loaded_policy.load_state_dict(checkpoint['policy_state_dict'])
    loaded_policy.eval()

    print(f"✓ Model loaded from {checkpoint_path}")
    print(f"  Best reward: {checkpoint['best_reward']:.2f}")

    return loaded_policy

# Example usage:
# loaded_policy = load_model('hierarchical_policy_qa.pt')

In [ ]:
# ===== NEW MARKDOWN CELL =====
"""
## 18. Test Week 7-8 Components

Validate energy-based value functions and multi-scale learning
"""

# ===== NEW CODE CELL - Test Energy Network =====
print("Testing Energy-Based Value Network...")
print("="*70)

# Create sample sequences
test_prompts = [
    "The quick brown fox jumps over the lazy dog.",
    "asdf jkl; qwer uiop zxcv bnm,",  # Random text
    "def calculate_sum(a, b): return a + b"  # Code
]

policy.eval()
with torch.no_grad():
    for i, prompt in enumerate(test_prompts):
        print(f"\nTest {i+1}: {prompt[:50]}...")

        # Encode
        token_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

        # Compute energy
        energy_result = policy.energy_network.compute_energy(token_ids)

        print(f"  Total Energy: {energy_result['total_energy'].item():.4f}")
        print(f"  Energy Value: {policy.energy_network.energy_to_value(energy_result).item():.4f}")
        print(f"  Avg Token Energy: {energy_result['token_energies'].mean().item():.4f}")

print("\n" + "="*70)
print("✓ Energy network test complete")

# ===== NEW CODE CELL - Test Multi-Scale Processing =====
print("\nTesting Multi-Scale Temporal Processing...")
print("="*70)

test_text = "This is a test sentence. It has multiple parts. Let's see how the model processes it at different scales."
token_ids = tokenizer.encode(test_text, return_tensors='pt').to(device)

policy.eval()
with torch.no_grad():
    outputs = policy(token_ids, return_full_output=True)

    print("\nMulti-Scale Outputs:")
    print(f"  Short-term shape: {outputs['multi_scale_output']['short_term'].shape}")
    print(f"  Medium-term shape: {outputs['multi_scale_output']['medium_term'].shape}")
    print(f"  Long-term shape: {outputs['multi_scale_output']['long_term'].shape}")
    print(f"  Structural shape: {outputs['multi_scale_output']['structural'].shape}")

    print("\nValue Estimates:")
    print(f"  High-level value: {outputs['high_value'].item():.4f}")
    print(f"  Low-level value: {outputs['low_value'].item():.4f}")
    print(f"  Energy value: {outputs['energy_value'].item():.4f}")
    print(f"  Multi-scale value: {outputs['multi_scale_value'].item():.4f}")
    print(f"  Fused value: {outputs['fused_value'].item():.4f}")

print("\n" + "="*70)
print("✓ Multi-scale processing test complete")

# ===== NEW CODE CELL - Test Energy-Guided Generation =====
print("\nTesting Energy-Guided Generation...")
print("="*70)

generator = EnergyBasedGenerator(policy.energy_network, tokenizer, device)

test_prompts = [
    "Question: What is 2+2? Answer:",
    "def fibonacci(n):",
    "The capital of France is"
]

for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    prompt_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    generated = generator.generate_low_energy(
        prompt_ids,
        max_length=30,
        num_candidates=5
    )

    generated_text = tokenizer.decode(generated[0])
    print(f"Generated: {generated_text}")

print("\n" + "="*70)
print("✓ Energy-guided generation test complete")

## Summary

This notebook implements a complete RL-LLM with hierarchical policy, energy-based values, and multi-scale learning:

**Week 5-6 Components:**
1. ✓ Multi-component reward system (fluency, coherence, task completion, safety)
2. ✓ Task-specific environments (Q&A, Code Generation)
3. ✓ Hierarchical policy (High-level intentions + Low-level tokens)
4. ✓ Hierarchical PPO training

**Week 7 Components:**
5. ✓ Energy-Based Value Network for text quality evaluation
6. ✓ Energy-guided text generation (find low-energy configurations)
7. ✓ Unified evaluation and generation through energy framework
8. ✓ Contrastive learning for structural quality

**Week 8 Components:**
9. ✓ Multi-Scale Temporal Processing (short/medium/long-term)
10. ✓ Hierarchical attention across time scales
11. ✓ Structural pattern detection (grammar discovery)
12. ✓ Multi-scale contrastive learning

**Training Features:**
- Progressive training with GAE (Generalized Advantage Estimation)
- Gradient clipping and mixed precision training
- Energy-based reward augmentation
- Multi-scale structural regularization
- Fused value estimation from 4 sources (hierarchical + energy + multi-scale)

**Architecture Integration:**
- Total parameters: ~30-35M (depends on vocab size)
- Full integration of hierarchical, energy-based, and multi-scale components
- Memory-optimized with gradient checkpointing
- Supports both Q&A and code generation tasks

**Next Steps (Week 9-10):**
- Scale up to larger models
- Comprehensive benchmark testing
- Analyze learned linguistic structure
- Prepare research paper (Week 11-12)